# Knesset Committee RAG Q&A
Chunk all sessions into 30-utterance windows → embed each utterance → mean-pool → vector index → Q&A with `Qwen/Qwen2.5-3B-Instruct`.

Runtime → **T4 GPU** (Kaggle: Accelerator → GPU T4 x2)

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install -q sentence-transformers transformers accelerate
!pip install -q -U bitsandbytes

In [ ]:
# ── Cell 2: Paths + config ────────────────────────────────────────────────────
import os, json

# ── Kaggle paths ──────────────────────────────────────────────────────────────
# Input dataset must be attached in Kaggle → Data → Add dataset
KAGGLE_INPUT_DIR  = '/kaggle/input/knesset-data'   # ← change to your dataset slug
KAGGLE_OUTPUT_DIR = '/kaggle/working'

# ── Persistent output dataset (for checkpoints) ───────────────────────────────
# One-time setup:
#   1. Go to kaggle.com → Datasets → New Dataset → name it "knesset-rag-index"
#   2. Set KAGGLE_USERNAME below
KAGGLE_USERNAME      = 'your-kaggle-username'       # ← change this
KAGGLE_OUTPUT_DATASET = 'knesset-rag-index'

os.chdir(KAGGLE_OUTPUT_DIR)
print('Working directory:', os.getcwd())

# ── Config ────────────────────────────────────────────────────────────────────
DATA_GLOB    = os.path.join(KAGGLE_INPUT_DIR, '**/*.json')
INDEX_PATH   = os.path.join(KAGGLE_OUTPUT_DIR, 'rag_index_dictabert.npz')
CHUNK_SIZE   = 30
CHUNK_STRIDE = 25
TOP_K        = 3
EMBED_MODEL  = 'dicta-il/dictabert'
LLM_MODEL    = 'Qwen/Qwen2.5-3B-Instruct'

In [ ]:
# ── Cell 3: Build / load chunk index (DictaBERT embedder) ────────────────────
import json, glob as _glob, numpy as np, torch, subprocess
from transformers import AutoTokenizer, AutoModel

CHECKPOINT_PATH  = os.path.join(KAGGLE_OUTPUT_DIR, 'rag_index_dictabert_checkpoint.npz')
CHECKPOINT_EVERY = 200  # save + push every N sessions

class DictaBERTEmbedder:
    """Mean-pool wrapper around dicta-il/dictabert (modern Hebrew BERT)."""
    def __init__(self, model_name=EMBED_MODEL, device='cpu', batch_size=32):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
        self.device     = device
        self.batch_size = batch_size

    def encode(self, texts, normalize=True):
        all_vecs = []
        for i in range(0, len(texts), self.batch_size):
            batch   = texts[i: i + self.batch_size]
            encoded = self.tokenizer(batch, padding=True, truncation=True,
                                     max_length=512, return_tensors='pt')
            encoded = {k: v.to(self.device) for k, v in encoded.items()}
            with torch.no_grad():
                out = self.model(**encoded)
            mask = encoded['attention_mask'].unsqueeze(-1).float()
            vecs = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            if normalize:
                vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)
            all_vecs.append(vecs.cpu().numpy())
        return np.concatenate(all_vecs, axis=0)

def load_session(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def make_chunks(utterances, size=CHUNK_SIZE, stride=CHUNK_STRIDE):
    return [utterances[i: i + size]
            for i in range(0, len(utterances), stride)
            if utterances[i: i + size]]

def _ensure_dataset_metadata():
    meta_path = os.path.join(KAGGLE_OUTPUT_DIR, 'dataset-metadata.json')
    if not os.path.exists(meta_path):
        with open(meta_path, 'w') as f:
            json.dump({
                'title': KAGGLE_OUTPUT_DATASET,
                'id': f'{KAGGLE_USERNAME}/{KAGGLE_OUTPUT_DATASET}',
                'licenses': [{'name': 'CC0-1.0'}],
            }, f)

def _dataset_exists():
    r = subprocess.run(
        ['kaggle', 'datasets', 'list', '--mine', '--search', KAGGLE_OUTPUT_DATASET],
        capture_output=True, text=True,
    )
    return KAGGLE_OUTPUT_DATASET in r.stdout

def push_to_kaggle_dataset(label='checkpoint'):
    _ensure_dataset_metadata()
    if _dataset_exists():
        cmd = ['kaggle', 'datasets', 'version', '-p', KAGGLE_OUTPUT_DIR,
               '-m', label, '--dir-mode', 'zip']
    else:
        cmd = ['kaggle', 'datasets', 'create', '-p', KAGGLE_OUTPUT_DIR,
               '--dir-mode', 'zip']
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  [kaggle push] {label} → {KAGGLE_USERNAME}/{KAGGLE_OUTPUT_DATASET}')
    else:
        print(f'  [kaggle push FAILED]\n{result.stderr.strip()}')

def save_checkpoint(all_vecs, all_meta, processed_paths, push=True):
    np.savez_compressed(
        CHECKPOINT_PATH,
        vecs=np.stack(all_vecs),
        metadata=np.array(all_meta, dtype=object),
        processed_paths=np.array(list(processed_paths)),
    )
    print(f'  [checkpoint] {len(all_meta)} chunks, {len(processed_paths)} sessions → {CHECKPOINT_PATH}')
    if push:
        push_to_kaggle_dataset(label=f'checkpoint-{len(processed_paths)}-sessions')

def load_checkpoint():
    candidates = [
        CHECKPOINT_PATH,
        os.path.join('/kaggle/input', KAGGLE_OUTPUT_DATASET, 'rag_index_dictabert_checkpoint.npz'),
    ]
    for path in candidates:
        if os.path.exists(path):
            data = np.load(path, allow_pickle=True)
            all_vecs   = list(data['vecs'])
            all_meta   = data['metadata'].tolist()
            done_paths = set(data['processed_paths'].tolist())
            print(f'  [checkpoint] resumed from {path}: {len(all_meta)} chunks, {len(done_paths)} sessions done')
            return all_vecs, all_meta, done_paths
    return [], [], set()

if os.path.exists(INDEX_PATH):
    print(f'Loading cached index from {INDEX_PATH} …')
    data     = np.load(INDEX_PATH, allow_pickle=True)
    vecs     = data['vecs']
    metadata = data['metadata'].tolist()
    print(f'Loaded {len(metadata)} chunks.')
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Building index with DictaBERT ({device}) …')
    embedder = DictaBERTEmbedder(device=device, batch_size=128)

    all_vecs, all_meta, done_paths = load_checkpoint()
    paths = sorted(_glob.glob(DATA_GLOB, recursive=True))
    remaining = [p for p in paths if p not in done_paths]
    print(f'Sessions total: {len(paths)}, remaining: {len(remaining)}, device: {device}')

    for p_idx, path in enumerate(remaining):
        if p_idx % 100 == 0:
            print(f'  {p_idx}/{len(remaining)} …')
        try:
            doc  = load_session(path)
            utts = doc.get('utterances', [])
            if not utts:
                done_paths.add(path)
                continue
            doc_id = os.path.splitext(os.path.basename(path))[0]
            title  = doc.get('title', '')
            date   = doc.get('date', doc.get('session_date', ''))[:10]
            url    = doc.get('url', doc.get('source_url', ''))

            texts    = [u.get('text', '').replace('\n', ' ').strip() for u in utts]
            utt_vecs = embedder.encode(texts)

            for i, chunk in enumerate(make_chunks(utts)):
                start     = i * CHUNK_STRIDE
                end       = start + len(chunk)
                chunk_vec = utt_vecs[start:end].mean(axis=0)
                chunk_vec /= (np.linalg.norm(chunk_vec) + 1e-9)

                all_vecs.append(chunk_vec.astype(np.float32))
                all_meta.append({
                    'doc_id': doc_id, 'title': title,
                    'date': date, 'url': url,
                    'from': int(chunk[0]['id']),
                    'to':   int(chunk[-1]['id']),
                    'utterances': chunk,
                })
            done_paths.add(path)
        except Exception as e:
            print(f'  [WARN] {path}: {e}')

        if (p_idx + 1) % CHECKPOINT_EVERY == 0:
            save_checkpoint(all_vecs, all_meta, done_paths, push=True)

    vecs     = np.stack(all_vecs)
    metadata = all_meta
    np.savez_compressed(INDEX_PATH, vecs=vecs,
                        metadata=np.array(metadata, dtype=object))
    push_to_kaggle_dataset(label='final-index')
    if os.path.exists(CHECKPOINT_PATH):
        os.remove(CHECKPOINT_PATH)
    print(f'Done: {len(metadata)} chunks → {INDEX_PATH}')

In [ ]:
# ── Cell 4: Load embedder (CPU) + LLM (4-bit, GPU) ────────────────────────────
import torch, gc
from transformers import pipeline, BitsAndBytesConfig

# reload on CPU so the LLM gets full VRAM
embedder = DictaBERTEmbedder(device='cpu')

def embed_query(text):
    return embedder.encode([text])[0].astype(np.float32)

gc.collect()
torch.cuda.empty_cache()
print(f'Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
pipe = pipeline(
    'text-generation',
    model=LLM_MODEL,
    model_kwargs={'quantization_config': quant, 'dtype': torch.bfloat16},
    device_map='auto',
)
print(f'LLM loaded. Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

In [ ]:
# ── Cell 5: RAG helpers ────────────────────────────────────────────────────────

SYSTEM_PROMPT = """אתה עוזר מחקר המתמחה בפרוטוקולי ועדות הכנסת.
ענה על שאלות המשתמש בעברית בלבד, בהתבסס אך ורק על הקטעים שסופקו.
בסוף התשובה ציין מאיזה ישיבות שאבת את המידע.
אם המידע אינו מופיע בקטעים — אמור זאת במפורש."""

def retrieve(query, k=TOP_K):
    q    = embed_query(query)
    sims = vecs @ q
    top  = np.argsort(sims)[::-1][:k]
    return [(metadata[i], float(sims[i])) for i in top]

def format_chunk(meta, sim):
    lines = [
        f"📄 {meta['title'][:70]}  [{meta['date']}]  אמירות {meta['from']}–{meta['to']}  (דמיון: {sim:.2f})",
    ]
    for u in meta['utterances']:
        speaker = u.get('speaker', '')
        text    = u.get('text', '').replace('\n', ' ').strip()
        lines.append(f"  [{u['id']}] {speaker}: {text}")
    return '\n'.join(lines)

def ask(question, k=TOP_K, max_new_tokens=1024):
    hits = retrieve(question, k)

    context_parts = []
    for meta, sim in hits:
        context_parts.append(format_chunk(meta, sim))

    context = '\n\n---\n\n'.join(context_parts)
    user_msg = (
        f"להלן קטעים רלוונטיים מפרוטוקולי ועדות הכנסת:\n\n"
        f"{context}\n\n"
        f"שאלה: {question}"
    )

    messages = [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + user_msg}]
    out      = pipe(messages, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out[0]['generated_text']
    answer    = generated[-1]['content'].strip() if isinstance(generated, list) else str(generated).strip()

    return answer, hits

def show(question, k=TOP_K):
    print(f'\n🔍 {question}\n')
    answer, hits = ask(question, k)
    print('─' * 60)
    print(answer)
    print('\n═ מקורות ═')
    for meta, sim in hits:
        url = meta.get('url', '')
        print(f"  • {meta['title'][:60]}  [{meta['date']}]  "
              f"אמירות {meta['from']}–{meta['to']}  sim={sim:.2f}")
        if url:
            print(f"    {url}")

print('RAG helpers ready.  Call show("השאלה שלך") to query.')

In [ ]:
# ── Cell 6: Interactive Q&A ───────────────────────────────────────────────────
# Run this cell repeatedly with different questions.

show("מה הוחלט בנושא צמצום השימוש במזומן?")

In [ ]:
# ── Cell 7: ipywidgets chat UI (optional) ─────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, HTML

txt = widgets.Text(placeholder='הקלד שאלה בעברית…', layout=widgets.Layout(width='80%'))
btn = widgets.Button(description='שאל', button_style='primary')
out = widgets.Output()

def on_ask(_):
    question = txt.value.strip()
    if not question:
        return
    with out:
        show(question)

btn.on_click(on_ask)
display(widgets.HBox([txt, btn]), out)